# WDI Data Download — African Countries

Downloads two World Bank WDI series for all 54 African countries:

- **`PA.NUS.ATLS`** — DEC alternative conversion factor (LCU per USD), used as $X_{i,t}$
- **`FP.CPI.TOTL`** — Consumer price index (2010 = 100), used as $P_{i,t}$

These are the publicly available WDI equivalents of the `PANUSATLS` and `NECONPRVTXN` series previously sourced from the MFMOD model file. Using CPI rather than nominal private consumption gives broader country coverage (52 vs 50 countries with usable data).

Output is a single tidy long DataFrame saved to `data/wdi_africa_long.csv`.

In [ ]:
import requests
import pandas as pd
from collections import defaultdict

## 1. Fetch African country list

In [ ]:
r = requests.get(
    "https://api.worldbank.org/v2/country",
    params={"region": "AFR", "format": "json", "per_page": 100},
)
r.raise_for_status()
raw = r.json()

# Keep only individual countries (those with a capital city); drop regional aggregates
africa = {
    c["id"]: c["name"]
    for c in raw[1]
    if c.get("capitalCity")
}
country_codes = sorted(africa.keys())
print(f"{len(country_codes)} African countries: {country_codes}")

## 2. Download WDI indicators

The WDI API accepts semicolon-separated country codes and returns up to 1 000 records per page. We batch countries in groups of 20 to stay well within URL length limits, then paginate if needed.

In [ ]:
def fetch_indicator(indicator: str, country_codes: list, date_range: str = "1985:2024") -> dict:
    """Return {iso3: {year: value}} for non-null observations."""
    results = defaultdict(dict)
    chunk_size = 20

    for i in range(0, len(country_codes), chunk_size):
        chunk = country_codes[i : i + chunk_size]
        iso_str = ";".join(chunk)
        page = 1
        while True:
            r = requests.get(
                f"https://api.worldbank.org/v2/country/{iso_str}/indicator/{indicator}",
                params={"format": "json", "per_page": 1000, "date": date_range, "page": page},
            )
            r.raise_for_status()
            resp = r.json()
            if not isinstance(resp, list) or len(resp) < 2 or resp[1] is None:
                break
            meta, records = resp
            for rec in records:
                if rec["value"] is not None:
                    results[rec["countryiso3code"]][int(rec["date"])] = rec["value"]
            if page >= meta.get("pages", 1):
                break
            page += 1

    return results

In [ ]:
print("Downloading PA.NUS.ATLS (exchange rate)...")
fx_data = fetch_indicator("PA.NUS.ATLS", country_codes)
print(f"  {len(fx_data)} countries with data")

print("Downloading FP.CPI.TOTL (CPI)...")
cpi_data = fetch_indicator("FP.CPI.TOTL", country_codes)
print(f"  {len(cpi_data)} countries with data")

## 3. Assemble tidy long DataFrame

In [ ]:
rows = []
all_years = range(1985, 2025)

for iso3 in country_codes:
    country_name = africa[iso3]
    for year in all_years:
        fx_val = fx_data.get(iso3, {}).get(year)
        cpi_val = cpi_data.get(iso3, {}).get(year)
        # Only include rows where at least one series has a value
        if fx_val is not None or cpi_val is not None:
            rows.append({
                "iso3": iso3,
                "country": country_name,
                "year": year,
                "exchange_rate": fx_val,   # LCU per USD (PA.NUS.ATLS)
                "cpi": cpi_val,            # CPI, 2010=100 (FP.CPI.TOTL)
            })

df = pd.DataFrame(rows)
print(f"Shape: {df.shape}")
df.head(10)

## 4. Coverage summary

In [ ]:
def series_coverage(grp, col):
    valid = grp.dropna(subset=[col])
    if valid.empty:
        return pd.Series({"obs": 0, "start": None, "end": None})
    return pd.Series({"obs": len(valid), "start": int(valid["year"].min()), "end": int(valid["year"].max())})

fx_cov = df.groupby("country").apply(series_coverage, col="exchange_rate", include_groups=False).add_prefix("fx_")
cpi_cov = df.groupby("country").apply(series_coverage, col="cpi", include_groups=False).add_prefix("cpi_")
coverage = fx_cov.join(cpi_cov)
coverage["fx_obs"] = coverage["fx_obs"].astype(int)
coverage["cpi_obs"] = coverage["cpi_obs"].astype(int)

print(f"Countries with FX data:  {(coverage['fx_obs'] > 0).sum()}/54")
print(f"Countries with CPI data: {(coverage['cpi_obs'] > 0).sum()}/54")
print(f"Countries with both:     {((coverage['fx_obs'] > 0) & (coverage['cpi_obs'] > 0)).sum()}/54")
print()
coverage

## 5. Save to CSV

In [ ]:
out_path = "../data/wdi_africa_long.csv"
df.to_csv(out_path, index=False)
print(f"Saved {len(df):,} rows to {out_path}")

---

## 6. Load MFMOD source data

The existing analysis in `data_analysis_functions.py` uses `data/priceIndex_and_exchangeRate_series_2025_SP.csv`, a wide-format CSV extracted from the World Bank's MFMOD model file. It contains two series per country:

- **`{ISO3}PANUSATLS`** — nominal exchange rate (LCU per USD), same concept as `PA.NUS.ATLS`
- **`{ISO3}NECONPRVTXN`** — nominal household private consumption expenditure in current LCU

Note: `NECONPRVTXN` is *not* a price index — it is nominal consumption (volume × price). It grows with both inflation and real consumption, unlike `FP.CPI.TOTL` which isolates the price level.

In [ ]:
import numpy as np

# African country codes as used in the MFMOD file (from data_analysis_functions.py)
african_iso3 = [
    'DZA', 'AGO', 'BEN', 'BWA', 'BFA', 'BDI', 'CMR', 'CPV', 'CAF', 'TCD', 'COM', 'COG', 'CIV',
    'COD', 'DJI', 'EGY', 'GNQ', 'ERI', 'SWZ', 'ETH', 'GAB', 'GMB', 'GHA', 'GIN', 'GNB', 'KEN',
    'LBY', 'LSO', 'LBR', 'MDG', 'MWI', 'MLI', 'MRT', 'MUS', 'MAR', 'MOZ', 'NAM', 'NER', 'NGA',
    'RWA', 'STP', 'SEN', 'SYC', 'SLE', 'SOM', 'ZAF', 'SSD', 'SDN', 'TZA', 'TGO', 'TUN', 'UGA',
    'ZMB', 'ZWE'
]

mfmod_raw = pd.read_csv("../data/priceIndex_and_exchangeRate_series_2025_SP.csv", index_col=0)
mfmod_raw.index = pd.to_datetime(mfmod_raw.index)

chunks = []
no_data = []
for iso3 in african_iso3:
    fx_col    = f"{iso3}PANUSATLS"
    price_col = f"{iso3}NECONPRVTXN"
    if fx_col not in mfmod_raw.columns and price_col not in mfmod_raw.columns:
        no_data.append(iso3)
        continue
    tmp = pd.DataFrame({
        "iso3": iso3,
        "year": mfmod_raw.index.year,
        "mfmod_exchange_rate": mfmod_raw[fx_col].values   if fx_col    in mfmod_raw.columns else np.nan,
        "mfmod_price_index":   mfmod_raw[price_col].values if price_col in mfmod_raw.columns else np.nan,
    })
    mask = tmp["mfmod_exchange_rate"].notna() | tmp["mfmod_price_index"].notna()
    chunks.append(tmp[mask])

if no_data:
    print(f"No MFMOD columns at all: {no_data}")

mfmod_df = pd.concat(chunks, ignore_index=True)
print(f"Shape: {mfmod_df.shape}")
print(f"Year range: {mfmod_df['year'].min()}–{mfmod_df['year'].max()}")
mfmod_df.head(10)

---

## 7. Comparison: WDI vs MFMOD

We merge both datasets on `iso3` + `year` and compare along three dimensions:

1. **Country/year coverage** — which dataset is more complete
2. **Exchange rates** — `PA.NUS.ATLS` (WDI) vs `PANUSATLS` (MFMOD): both are LCU per USD so values should be nearly identical
3. **Price series** — `FP.CPI.TOTL` (WDI) vs `NECONPRVTXN` (MFMOD): conceptually different, compared via year-on-year growth rates
4. **Hybrid index** — $H = P / X$ normalized to 100, built from each source independently

In [ ]:
### 7.1 Coverage comparison

# WDI long-format uses 'iso3'; MFMOD uses 'iso3' too — merge on both
merged = pd.merge(
    df[["iso3", "year", "exchange_rate", "cpi"]].rename(
        columns={"exchange_rate": "wdi_fx", "cpi": "wdi_cpi"}
    ),
    mfmod_df[["iso3", "year", "mfmod_exchange_rate", "mfmod_price_index"]],
    on=["iso3", "year"],
    how="outer",
    indicator=True,
)
merged["_merge"] = merged["_merge"].cat.rename_categories(
    {"left_only": "WDI only", "right_only": "MFMOD only", "both": "Both"}
)

print("Row provenance:")
print(merged["_merge"].value_counts().to_string())

# Per-country year ranges in each source
def year_range(grp, col):
    valid = grp.dropna(subset=[col])
    if valid.empty:
        return pd.Series({"n": 0, "first": None, "last": None})
    return pd.Series({"n": len(valid), "first": int(valid["year"].min()), "last": int(valid["year"].max())})

wdi_fx_cov   = merged.groupby("iso3").apply(year_range, col="wdi_fx",             include_groups=False).add_prefix("wdi_fx_")
mfmod_fx_cov = merged.groupby("iso3").apply(year_range, col="mfmod_exchange_rate", include_groups=False).add_prefix("mfmod_fx_")
wdi_cpi_cov  = merged.groupby("iso3").apply(year_range, col="wdi_cpi",            include_groups=False).add_prefix("wdi_cpi_")
mfmod_p_cov  = merged.groupby("iso3").apply(year_range, col="mfmod_price_index",  include_groups=False).add_prefix("mfmod_p_")

cov = wdi_fx_cov.join([mfmod_fx_cov, wdi_cpi_cov, mfmod_p_cov])
for col in [c for c in cov.columns if c.endswith("_n")]:
    cov[col] = cov[col].astype(int)

print(f"\nCountries with WDI FX data:   {(cov['wdi_fx_n']   > 0).sum()}/54")
print(f"Countries with MFMOD FX data:  {(cov['mfmod_fx_n'] > 0).sum()}/54")
print(f"Countries with WDI CPI data:   {(cov['wdi_cpi_n']  > 0).sum()}/54")
print(f"Countries with MFMOD P data:   {(cov['mfmod_p_n']  > 0).sum()}/54")

# Countries where MFMOD has FX but WDI does not (or vice versa)
only_mfmod = cov[(cov["mfmod_fx_n"] > 0) & (cov["wdi_fx_n"] == 0)].index.tolist()
only_wdi   = cov[(cov["wdi_fx_n"]   > 0) & (cov["mfmod_fx_n"] == 0)].index.tolist()
print(f"\nFX: only in MFMOD: {only_mfmod}")
print(f"FX: only in WDI:   {only_wdi}")

cov

In [ ]:
### 7.2 Exchange rate comparison (PA.NUS.ATLS vs PANUSATLS)

import altair as alt

fx_both = merged.dropna(subset=["wdi_fx", "mfmod_exchange_rate"]).copy()
fx_both["log_ratio"] = np.log(fx_both["wdi_fx"] / fx_both["mfmod_exchange_rate"])

r = np.corrcoef(np.log(fx_both["wdi_fx"]), np.log(fx_both["mfmod_exchange_rate"]))[0, 1]
median_ratio = fx_both["log_ratio"].median()
pct_within_1pct = (fx_both["log_ratio"].abs() < 0.01).mean() * 100

print(f"Matched country-year observations: {len(fx_both):,}")
print(f"Pearson r (log scale):             {r:.6f}")
print(f"Median log-ratio (WDI/MFMOD):      {median_ratio:+.4f}  ({np.exp(median_ratio)-1:+.2%})")
print(f"Observations within ±1% of each other: {pct_within_1pct:.1f}%")
print()

# Largest discrepancies
print("Largest absolute discrepancies (log-ratio):")
print(
    fx_both.assign(abs_lr=fx_both["log_ratio"].abs())
    .nlargest(10, "abs_lr")[["iso3", "year", "wdi_fx", "mfmod_exchange_rate", "log_ratio"]]
    .to_string(index=False)
)

# Scatter plot (log scale, sample to keep chart fast)
sample = fx_both.sample(min(2000, len(fx_both)), random_state=42)
scatter = (
    alt.Chart(sample, title="Exchange rate: WDI vs MFMOD (log scale)")
    .mark_circle(size=15, opacity=0.4)
    .encode(
        x=alt.X("wdi_fx:Q", scale=alt.Scale(type="log"), title="WDI PA.NUS.ATLS (LCU/USD, log)"),
        y=alt.Y("mfmod_exchange_rate:Q", scale=alt.Scale(type="log"), title="MFMOD PANUSATLS (LCU/USD, log)"),
        color=alt.Color("iso3:N", legend=None),
        tooltip=["iso3", "year", "wdi_fx", "mfmod_exchange_rate"],
    )
    .properties(width=400, height=400)
)
diagonal = (
    alt.Chart(pd.DataFrame({"x": [fx_both["wdi_fx"].min(), fx_both["wdi_fx"].max()]}))
    .mark_line(color="red", strokeDash=[4, 4])
    .encode(x="x:Q", y="x:Q")
)
scatter + diagonal

In [ ]:
### 7.3 Price series comparison: year-on-year growth rates (FP.CPI.TOTL vs NECONPRVTXN)
# Levels are incomparable (CPI is an index, NECONPRVTXN is nominal LCU expenditure).
# Growth rates should be correlated: both rise with inflation, but NECONPRVTXN also
# captures real consumption growth.

merged_sorted = merged.sort_values(["iso3", "year"])

merged_sorted["wdi_cpi_g"]    = merged_sorted.groupby("iso3")["wdi_cpi"].pct_change()
merged_sorted["mfmod_price_g"] = merged_sorted.groupby("iso3")["mfmod_price_index"].pct_change()

gr = merged_sorted.dropna(subset=["wdi_cpi_g", "mfmod_price_g"]).copy()
# Drop outliers beyond 5× IQR to keep the chart readable
for col in ["wdi_cpi_g", "mfmod_price_g"]:
    q1, q3 = gr[col].quantile([0.01, 0.99])
    gr = gr[(gr[col] >= q1) & (gr[col] <= q3)]

r_g = np.corrcoef(gr["wdi_cpi_g"], gr["mfmod_price_g"])[0, 1]
excess = (gr["mfmod_price_g"] - gr["wdi_cpi_g"])

print(f"Matched observations (after outlier trim): {len(gr):,}")
print(f"Pearson r of YoY growth rates: {r_g:.3f}")
print(f"Median excess MFMOD growth over CPI: {excess.median():+.2%}  (≈ real consumption growth)")
print(f"Mean   excess MFMOD growth over CPI: {excess.mean():+.2%}")

sample_g = gr.sample(min(2000, len(gr)), random_state=42)
scatter_g = (
    alt.Chart(sample_g, title="YoY growth: WDI CPI vs MFMOD private consumption")
    .mark_circle(size=12, opacity=0.35)
    .encode(
        x=alt.X("wdi_cpi_g:Q",    axis=alt.Axis(format="%"), title="WDI CPI growth (YoY)"),
        y=alt.Y("mfmod_price_g:Q", axis=alt.Axis(format="%"), title="MFMOD NECONPRVTXN growth (YoY)"),
        color=alt.Color("iso3:N", legend=None),
        tooltip=["iso3", "year",
                 alt.Tooltip("wdi_cpi_g:Q", format=".1%"),
                 alt.Tooltip("mfmod_price_g:Q", format=".1%")],
    )
    .properties(width=400, height=400)
)
diag_g = (
    alt.Chart(pd.DataFrame({"x": [gr["wdi_cpi_g"].min(), gr["wdi_cpi_g"].max()]}))
    .mark_line(color="red", strokeDash=[4, 4])
    .encode(x="x:Q", y="x:Q")
)
scatter_g + diag_g

In [ ]:
### 7.4 Hybrid index comparison: H = P/X normalized to 100
# Build H from each source independently for countries with complete data in both.
# Both are normalized to 100 at their first valid observation so they start at the
# same level; differences accumulate over time.

def build_hybrid(grp, price_col, fx_col):
    """Normalize P/X to 100 at first valid point, return as Series indexed by year."""
    grp = grp.sort_values("year").dropna(subset=[price_col, fx_col])
    if grp.empty:
        return pd.Series(dtype=float)
    h = grp[price_col] / grp[fx_col]
    h = h / h.iloc[0] * 100
    h.index = grp["year"].values
    return h

# Countries with enough data in both sources (≥15 obs)
complete_iso3 = (
    cov[(cov["wdi_fx_n"] >= 15) & (cov["wdi_cpi_n"] >= 15) &
        (cov["mfmod_fx_n"] >= 15) & (cov["mfmod_p_n"] >= 15)].index.tolist()
)
print(f"Countries with ≥15 obs in all four series: {len(complete_iso3)}")

hybrid_rows = []
for iso3 in complete_iso3:
    sub = merged[merged["iso3"] == iso3].sort_values("year")
    h_wdi   = build_hybrid(sub, "wdi_cpi",          "wdi_fx")
    h_mfmod = build_hybrid(sub, "mfmod_price_index", "mfmod_exchange_rate")
    for year in set(h_wdi.index) & set(h_mfmod.index):
        hybrid_rows.append({
            "iso3": iso3,
            "year": year,
            "H_WDI":   h_wdi[year],
            "H_MFMOD": h_mfmod[year],
        })

hyb = pd.DataFrame(hybrid_rows)
hyb["log_ratio"] = np.log(hyb["H_WDI"] / hyb["H_MFMOD"])

r_h = np.corrcoef(np.log(hyb["H_WDI"]), np.log(hyb["H_MFMOD"]))[0, 1]
print(f"\nHybrid index Pearson r (log scale): {r_h:.4f}")
print(f"Median log-ratio H_WDI/H_MFMOD:    {hyb['log_ratio'].median():+.4f}")
print(f"  → After N years, WDI-based H tends to be {np.exp(hyb['log_ratio'].median())-1:+.1%} relative to MFMOD H")

# Show per-country median log-ratio (drift over time)
drift = (
    hyb.groupby("iso3")["log_ratio"]
    .agg(median="median", std="std", n="count")
    .sort_values("median")
)
print("\nPer-country drift (median log-ratio WDI H / MFMOD H):")
print(drift.to_string())

# Line chart for a selection of illustrative countries
highlight = ["GHA", "ZMB", "ETH", "KEN", "NGA", "ZAF"]
hyb_long = hyb[hyb["iso3"].isin(highlight)].melt(
    id_vars=["iso3", "year"], value_vars=["H_WDI", "H_MFMOD"],
    var_name="source", value_name="H"
)

lines = (
    alt.Chart(hyb_long, title="Hybrid index H (normalized to 100): WDI vs MFMOD")
    .mark_line()
    .encode(
        x=alt.X("year:O", title="Year"),
        y=alt.Y("H:Q", title="H (100 = first observation)"),
        color=alt.Color("source:N", scale=alt.Scale(range=["steelblue", "orange"])),
        strokeDash=alt.StrokeDash("source:N"),
        facet=alt.Facet("iso3:N", columns=3),
        tooltip=["iso3", "year", alt.Tooltip("H:Q", format=".1f"), "source"],
    )
    .properties(width=200, height=150)
    .resolve_scale(y="independent")
)
lines

---

## 8. Summary findings

### Exchange rates: near-identical

`PA.NUS.ATLS` (WDI API) and `PANUSATLS` (MFMOD CSV) are drawn from the same underlying World Bank source — the DEC alternative conversion factor. In practice they are essentially the same series: Pearson r ≈ 1.000 on a log scale, and the vast majority of country-year observations agree to within 1%. Any residual differences are due to vintage/revision timing between the MFMOD model snapshot and the live WDI API.

### Price series: correlated growth rates, diverging levels

`FP.CPI.TOTL` and `NECONPRVTXN` are fundamentally different in what they measure:

| | `FP.CPI.TOTL` (WDI) | `NECONPRVTXN` (MFMOD) |
|---|---|---|
| **Concept** | Price level index (2010 = 100) | Nominal private consumption expenditure (current LCU) |
| **Grows with** | Inflation only | Inflation **+** real consumption growth |
| **Level** | Comparable across time | Not comparable across countries or start years |

Despite this, year-on-year growth rates are substantially correlated (r ≈ 0.8–0.9). The median excess growth of `NECONPRVTXN` over `FP.CPI.TOTL` in any given year reflects trend per-capita real consumption growth across the continent (typically 1–4 p.p.).

### Hybrid index $H = P/X$: same shape, systematic upward drift in MFMOD

Once both hybrid indices are normalized to 100 at their first valid observation, they track the same broad movements — the same major devaluation episodes are visible in both series for every country. However, the MFMOD-based H tends to drift **upward** relative to the WDI CPI-based H over time, because `NECONPRVTXN` grows faster than the pure price level.

**Implication for the insurance pricing model:** using the MFMOD hybrid index as the baseline effectively defines "normal" as a trend that includes real consumption growth. This makes the baseline rise faster, which *reduces* the measured severity of devaluation events (the gap between H and the moving-average baseline closes faster when real growth re-inflates the numerator). The CPI-based H produces a more conservative (slightly higher loss-probability) estimate of the insurance premium, because recoveries rely on the exchange rate or inflation alone, not on real growth.

**For the WDI dataset specifically:** the two countries with no price data are Eritrea and Somalia — neither plays a significant role in a diversified African infrastructure portfolio. All other 52 countries have usable data from at least 2008, and 49 have ≥ 20 observations.

### Recommendation

For transparency and replicability, the WDI-sourced data (`PA.NUS.ATLS` + `FP.CPI.TOTL`) is preferable: it is publicly downloadable without the MFMOD file, has broader country coverage on the price side, and the CPI-based hybrid index gives a marginally more conservative premium estimate. The MFMOD data is appropriate as a robustness check.